# Session 2: Data Wrangling with pandas

**Introduction to Data Analysis with Python**

---

### What this session covers

| Part | Topic | Time |
|------|-------|------|
| A | Loading and inspecting two datasets | 10 min |
| B | Cleaning and recoding variables | 20 min |
| C | Grouped summaries and merging files | 20 min |
| D | Exercises | 10 min |

### Learning outcomes

By the end of this session you will be able to:

- Rename columns and standardise their values
- Filter rows and select columns
- Handle missing values and invalid data
- Create new variables and recode existing ones
- Produce grouped summary statistics
- Merge two DataFrames on a shared key

### Datasets used in this session

| File | What it contains |
|------|-----------------|
| `ukhls_individual.csv` | 520 individuals — demographics, employment, income, health |
| `ukhls_household.csv` | 280 households — region, tenure, number of rooms |

These files share a household identifier (`hidp`) and can be linked together — exactly like Stata's `merge 1:m hidp using household.dta`.

> **Note:** these are synthetic datasets with the same structure as the UK Household Longitudinal Study (Understanding Society). They have been designed with realistic messiness to give you practice cleaning real-world data.

---

## Part A: Loading and Inspecting Both Files

In [ ]:
import pandas as pd
import numpy as np

# Load both files
ind = pd.read_csv("ukhls_individual.csv")
hh  = pd.read_csv("ukhls_household.csv")

print(f"Individual file: {ind.shape[0]} rows, {ind.shape[1]} columns")
print(f"Household file:  {hh.shape[0]} rows, {hh.shape[1]} columns")

In [ ]:
ind.head()

In [ ]:
hh.head()

In [ ]:
# Check data types and missing values in each file
print("=== Individual file ===")
ind.info()
print()
print("=== Household file ===")
hh.info()

In [ ]:
# Count missing values column by column
print("Missing values — individual file:")
print(ind.isnull().sum())
print()
print("Missing values — household file:")
print(hh.isnull().sum())

> **What to notice:** `income_monthly` and `smoker` have some missing values, and
> `hours_worked` contains `-9` — a common Stata convention meaning "not applicable".
> Part B deals with all of these.

---

## Part B: Cleaning and Recoding Variables

### B.1 Renaming columns

Pandas column names with spaces or mixed case are awkward to type repeatedly.
Converting to `snake_case` is standard practice.

```stata
* Stata — you can't easily rename all columns at once
rename income_monthly income
```

In [ ]:
# Rename columns to lowercase snake_case where needed
# (these are already reasonable, but let's tidy up 'health' to be explicit)
ind = ind.rename(columns={"health": "health_rating"})
print("Columns:", list(ind.columns))

### B.2 Standardising a messy categorical variable

The `sex` column was coded inconsistently — look at what's in it:

In [ ]:
# What values are actually in the sex column?
ind["sex"].value_counts()

In [ ]:
# Standardise to 'Male' / 'Female' using str.lower() and a mapping
sex_clean = ind["sex"].str.strip().str.lower()
sex_clean = sex_clean.map({"male": "Male", "m": "Male", "female": "Female"})

# Check the result before committing
print("Before:", ind["sex"].value_counts().to_dict())
print("After: ", sex_clean.value_counts().to_dict())
print("Missing after cleaning:", sex_clean.isnull().sum())

In [ ]:
# Assign the cleaned column back
ind["sex"] = sex_clean

### B.3 Handling invalid numeric values

In the source data, `hours_worked = -9` means 'not applicable' (the person is not in paid employment).
We should replace -9 with `NaN` so pandas treats it as missing rather than as a real number.

```stata
* Stata
replace hours_worked = . if hours_worked == -9
```

In [ ]:
# Replace -9 with NaN
ind["hours_worked"] = ind["hours_worked"].replace(-9, np.nan)

# Verify
print("Min hours_worked:", ind["hours_worked"].min())
print("Missing:         ", ind["hours_worked"].isnull().sum())

### B.4 Dropping and filling missing values

There are two strategies:
- **Drop** rows with missing values (`dropna`) — use when missingness is rare and the missing rows aren't needed
- **Fill** with a default value (`fillna`) — use when you have a principled reason for the fill value

```stata
* Stata
drop if income_monthly == .
```

In [ ]:
# How many rows would we lose if we dropped all rows missing income?
n_before = len(ind)
n_after  = ind.dropna(subset=["income_monthly"]).shape[0]
print(f"Rows before: {n_before}")
print(f"Rows after:  {n_after}  ({n_before - n_after} dropped)")

In [ ]:
# For this analysis we'll keep missing income rows but note how to fill if needed:
# ind["income_monthly"] = ind["income_monthly"].fillna(ind["income_monthly"].median())

# Instead, just confirm what we have
print(ind["income_monthly"].describe())

### B.5 Creating new variables

```stata
* Stata
gen age_group = "Young adult" if age < 35
replace age_group = "Middle-aged" if age >= 35 & age < 60
replace age_group = "Older adult" if age >= 60
```

In [ ]:
# Create an age group variable using pd.cut()
ind["age_group"] = pd.cut(
    ind["age"],
    bins=[17, 34, 59, 90],
    labels=["Young adult (18-34)", "Middle-aged (35-59)", "Older adult (60+)"]
)

ind["age_group"].value_counts()

In [ ]:
# Create a flag for full-time workers (35+ hours per week)
# np.where(condition, value_if_true, value_if_false)  <-- like Stata's gen + replace
ind["full_time"] = np.where(ind["hours_worked"] >= 35, 1, 0)

# Anyone with missing hours_worked will get 0 — flag that
ind.loc[ind["hours_worked"].isnull(), "full_time"] = np.nan
print(ind["full_time"].value_counts(dropna=False))

---

## Part C: Grouped Summaries and Merging Files

### C.1 Grouped summary statistics

```stata
* Stata
collapse (mean) income_monthly, by(employment)
```

In [ ]:
# Mean income by employment status
(ind.groupby("employment")["income_monthly"]
    .agg(["mean", "median", "count"])
    .round(0)
    .sort_values("mean", ascending=False))

In [ ]:
# Multiple columns at once
ind.groupby("employment")[["income_monthly", "hours_worked"]].mean().round(1)

In [ ]:
# groupby with a categorical variable
ind.groupby("age_group")["health_rating"].mean().round(2)

> **Interpreting health_rating:** 1 = Excellent, 5 = Poor.
> A higher mean indicates worse average health — which is what we'd expect to rise with age.

### C.2 Merging two DataFrames

```stata
* Stata
merge m:1 hidp using "ukhls_household.dta"
```

In [ ]:
# Merge individual file (many rows) with household file (one row per household)
# 'how="left"' keeps all individuals and adds household info where it exists
merged = pd.merge(ind, hh, on="hidp", how="left")

print(f"Individual rows: {len(ind)}")
print(f"Merged rows:     {len(merged)}")
print(f"New columns:     {[c for c in merged.columns if c not in ind.columns]}")

In [ ]:
# Check the merge worked — every individual should now have a region
print("Missing region after merge:", merged["region"].isnull().sum())
merged[["pidp", "hidp", "age", "sex", "region", "tenure"]].head(10)

In [ ]:
# Now we can cross-tabulate individual characteristics with household-level variables
# e.g. mean income by region
merged.groupby("region")["income_monthly"].mean().round(0).sort_values(ascending=False)

---

## Quick Reference: Stata to pandas

| Task | Stata | Python / pandas |
|------|-------|-----------------|
| Rename column | `rename old new` | `df.rename(columns={"old": "new"})` |
| Keep columns | `keep var1 var2` | `df[["var1", "var2"]]` |
| Filter rows | `keep if age > 18` | `df[df["age"] > 18]` |
| Replace invalid | `replace x = . if x == -9` | `df["x"].replace(-9, np.nan)` |
| Create variable | `gen y = x * 2` | `df["y"] = df["x"] * 2` |
| Conditional gen | `gen flag = 1 if ...` | `np.where(condition, 1, 0)` |
| Recode to groups | `recode age (18/34=1)` | `pd.cut(df["age"], bins, labels)` |
| Grouped mean | `collapse (mean) y, by(g)` | `df.groupby("g")["y"].mean()` |
| Merge datasets | `merge m:1 id using b` | `pd.merge(a, b, on="id", how="left")` |
| Drop missing | `drop if x == .` | `df.dropna(subset=["x"])` |
| Fill missing | `replace x = 0 if x == .` | `df["x"].fillna(0)` |
| Missing count | `misstable summarize` | `df.isnull().sum()` |

---

## Part D: Exercises

---

### Exercise 1 — Explore employment

Using the cleaned `ind` DataFrame:

1. How many people are in each employment category?
2. What is the median income for each employment category?
3. What proportion of employed people are full-time workers?

*(Hint for Q3: filter to employed rows first, then use `.mean()` on the `full_time` column.)*

In [ ]:
# Exercise 1 — your code here

---

### Exercise 2 — Recode health

The `health_rating` column uses a 1–5 scale (1 = Excellent, 5 = Poor).

Create a new column called `health_good` that is:
- `1` if `health_rating` is 1, 2, or 3
- `0` if `health_rating` is 4 or 5

Then calculate what proportion of the sample reports good health.

In [ ]:
# Exercise 2 — your code here

---

### Exercise 3 — Merge and compare

Using the `merged` DataFrame (individual + household):

1. What is the most common tenure type?
2. Does mean income differ between renters and owners? (Group by `tenure`.)
3. Which region has the highest proportion of individuals in full-time work?

*(Hint for Q3: `groupby("region")["full_time"].mean()` gives you the proportion directly, since `full_time` is coded 0/1.)*

In [ ]:
# Exercise 3 — your code here

---

### Exercise 4 — Stretch goal: chain it together

Write a single chained expression (using `.pipe()` or method chaining) that:
1. Starts from `ind`
2. Filters to individuals aged 25–55
3. Selects only `sex`, `employment`, `income_monthly`, and `age_group`
4. Drops rows with missing `income_monthly`
5. Groups by `sex` and `employment` and computes mean income
6. Rounds to the nearest pound

*(This is harder — don't worry if it takes a few attempts.)*

In [ ]:
# Exercise 4 — your code here

---

## Wrap-up

### What you covered today

- Loading and inspecting two related CSV files
- Standardising a messy categorical variable with `.str` methods and `.map()`
- Replacing invalid values with `NaN`
- Handling missing data with `dropna` and `fillna`
- Creating new columns with `np.where()` and `pd.cut()`
- Producing grouped summaries with `groupby()`
- Merging two DataFrames on a shared key with `pd.merge()`

### Coming up in Session 3

Session 3 moves from wrangling to **visualisation**. You will use matplotlib and seaborn to produce publication-quality charts from the World Development Indicators dataset, and see how the CRAFT framework applies to the analysis and transform steps of a data workflow.

### Save your work

**File → Save Notebook** (or Cmd+S).